# Scoring de credito: quien no paga y por que

**Que responde este cuaderno**

1. Cual es la tasa de impago de la cartera.
2. Como cambia el riesgo segun el estado de la cuenta, el proposito y el historial.
3. Que variables pesan mas en un modelo simple, en odds ratio.
4. Cuanto discrimina el modelo (AUC) y donde se equivoca.
5. Como mueven el riesgo el plazo y el monto.

**Datos**: `german_credit_cleaned.csv` (1.000 creditos, dataset German Credit). Se descarga por URL al ejecutar: no hace falta subir nada.

**Como correrlo**: Runtime > Run all. pandas, matplotlib y scikit-learn.

In [ ]:
import io, pathlib, urllib.request
import pandas as pd
import matplotlib.pyplot as plt

BASE = 'https://raw.githubusercontent.com/Nicolenki7/German_Credit_Scoring_Data/main/'
pathlib.Path('outputs').mkdir(exist_ok=True)

with urllib.request.urlopen(BASE + 'german_credit_cleaned.csv') as r:
    df = pd.read_csv(io.BytesIO(r.read()))

print('filas:', len(df), '| columnas:', len(df.columns))
print('objetivo:', df['target'].value_counts().to_dict())
df.head(3)

## 1. Tasa de impago

La linea base del negocio: cualquier modelo tiene que mejorarla.

In [ ]:
impago = (df['target'] == 'bad').mean() * 100
ratio = (df['target'] == 'good').sum() / (df['target'] == 'bad').sum()
print(f'RESPUESTA: el {impago:.1f} % de los {len(df)} creditos termina en impago, {ratio:.2f} buenos por cada malo.')

## 2. Donde esta el riesgo

Comparamos categorias con 30 o mas creditos, para que la diferencia no sea ruido de muestra.

In [ ]:
def tasa_por(col, min_n=30):
    g = df.groupby(col)['target'].agg(n='count', impago=lambda s: (s == 'bad').mean() * 100)
    g = g[g['n'] >= min_n].sort_values('impago', ascending=False)
    g['impago'] = g['impago'].round(1)
    return g

for col in ['checking_acc_status', 'purpose', 'cred_hist']:
    print('==', col)
    print(tasa_por(col).head(6).to_string())
    print()

print(f'RESPUESTA: la media es {impago:.1f} % y el estado de la cuenta corriente es lo que mas separa: sus peores categorias casi duplican la media.')

## 3. Que pesa en el modelo

Regresion logistica con one-hot para categoricas y estandarizado para numericas. Los coeficientes se convierten a **odds ratio**: por encima de 1 sube el riesgo de impago, por debajo lo baja.

In [ ]:
import math
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = df.drop(columns=['target'])
y = (df['target'] == 'bad').astype(int)
cat = X.select_dtypes(include='object').columns.tolist()
num = [c for c in X.columns if c not in cat]

prep = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat),
    ('num', StandardScaler(), num),
])
modelo = Pipeline([('prep', prep), ('clf', LogisticRegression(max_iter=2000, random_state=42))])
modelo.fit(X, y)

nombres = modelo.named_steps['prep'].get_feature_names_out()
coefs = modelo.named_steps['clf'].coef_[0]
pesos = pd.DataFrame({'variable': nombres, 'odds_ratio': [round(math.exp(v), 2) for v in coefs]})
pesos = pesos.sort_values('odds_ratio', ascending=False).reset_index(drop=True)
pesos.to_csv('outputs/odds_ratios.csv', index=False)

print('Suben el riesgo:')
print(pesos.head(8).to_string(index=False))
print()
print('Lo bajan:')
print(pesos.tail(5).to_string(index=False))
print()
primero = pesos['variable'].iloc[0]
print(f'RESPUESTA: la variable con mas peso al alza es {primero}.')

## 4. Sirve el modelo?

AUC sobre una particion reservada: mide ordenar bien el riesgo, no acertar un umbral concreto.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
modelo.fit(X_tr, y_tr)
prob = modelo.predict_proba(X_te)[:, 1]
auc = roc_auc_score(y_te, prob)

print(f'RESPUESTA: AUC = {auc:.3f} sobre {len(y_te)} creditos reservados.')
print()
print('Matriz de confusion con umbral 0,5:')
print(confusion_matrix(y_te, (prob >= 0.5).astype(int)))
print(classification_report(y_te, (prob >= 0.5).astype(int), target_names=['paga', 'impaga']))
print('Nota honesta: con 1.000 filas el AUC varia varios puntos entre particiones; el numero de arriba es de una sola.')

## 5. Plazo y monto

Las dos variables continuas que un comite de credito mira siempre.

In [ ]:
df['tramo_plazo'] = pd.cut(df['duration'], bins=[0, 12, 24, 36, 72], labels=['<=12m', '13-24m', '25-36m', '>36m'])
df['tramo_monto'] = pd.qcut(df['loan_amt'], 4, labels=['Q1 bajo', 'Q2', 'Q3', 'Q4 alto'])

by_plazo = df.groupby('tramo_plazo', observed=True)['target'].agg(n='count', impago=lambda s: round((s == 'bad').mean() * 100, 1))
by_monto = df.groupby('tramo_monto', observed=True)['target'].agg(n='count', impago=lambda s: round((s == 'bad').mean() * 100, 1))

print(by_plazo.to_string())
print()
print(by_monto.to_string())

peor_tramo = by_plazo['impago'].idxmax()
mejor_valor = by_plazo['impago'].min()
print()
print(f'RESPUESTA: el tramo {peor_tramo} tiene {by_plazo.loc[peor_tramo, "impago"]} % de impago frente al {mejor_valor} % del mejor tramo.')

ax = by_plazo['impago'].plot(kind='bar', figsize=(6, 3), rot=0)
ax.set_ylabel('% impago')
ax.set_title('Impago por plazo del credito')
plt.tight_layout()
plt.savefig('outputs/impago_por_plazo.png', dpi=150)
plt.show()

## Limites

- Dataset historico aleman de 1.000 filas: sirve para el metodo, no para decidir credito real.
- No hay coste asimetrico: negar a un buen pagador cuesta distinto que aprobar un impago.
- Sin validacion temporal ni calibracion de probabilidades.
- Las categorias vienen ya limpias; en un caso real habria que auditar esa codificacion.